Understanding how Spark is structured under the hood is essential for writing efficient code, diagnosing performance problems, and making sense of the Spark UI. This notebook covers the physical architecture of a Spark cluster and the logical execution model — how your code becomes tasks running on machines.

## The Cluster Architecture

Every Spark application involves three types of process:

![Spark Cluster Architecture](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-cluster-architecture.svg)

| Process | Where it runs | Responsibility |
|---|---|---|
| **Driver** | One dedicated node (or your laptop in local mode) | Runs your code, builds the execution plan, coordinates everything |
| **Cluster Manager** | Separate service | Allocates CPU and memory across the cluster |
| **Executor** | One JVM per worker node | Executes tasks, stores cached data |

## The Driver

The Driver is the brain of a Spark application. It is the process that runs your `main()` function (or your notebook/shell session) and hosts the `SparkSession`.

**What the Driver does:**
- Parses your transformations and builds a logical plan
- Hands the plan to **Catalyst** to produce an optimized physical plan
- **DAG Scheduler** — breaks the physical plan into a graph of stages
- **Task Scheduler** — assigns individual tasks to free executor slots, respecting data locality
- Collects results when you call an action that returns data to the driver (e.g., `.collect()`, `.take()`)

> The Driver must stay alive for the duration of the application. If it dies, the application fails. For production jobs, run the driver on a stable node — not on your laptop.

## The Cluster Manager

The Cluster Manager is a separate service that Spark talks to in order to acquire resources. Spark is intentionally **agnostic** about which cluster manager you use — it just asks for N cores and M GB of memory, and the cluster manager decides where to provision them.

| Cluster Manager | When to use |
|---|---|
| **Standalone** | Simplest option; ships with Spark; good for dedicated Spark clusters |
| **YARN** | Standard in Hadoop ecosystems (CDH, HDP, Amazon EMR) |
| **Kubernetes** | Cloud-native; each executor runs in its own pod; preferred for new deployments |
| **Databricks** | Fully managed; abstracts the cluster manager entirely |
| **Mesos** | Legacy; largely replaced by Kubernetes |

On **Databricks** or **local mode**, you don't interact with the cluster manager directly.

## Executors

An Executor is a **JVM process** launched on a worker node at the start of an application. It stays alive for the entire application lifetime (unless it crashes).

Each executor has:
- A fixed number of **CPU cores** — each core can run one task at a time
- A fixed amount of **heap memory** — split between execution memory and storage memory
- **Local disk** — for spilling data that doesn't fit in RAM

Key points:
- Executors are **isolated** — one application's executors cannot see another's data
- Cached data lives in executor memory (or disk) — it disappears when the application ends
- If an executor crashes, Spark reschedules its tasks on other executors using RDD lineage

## Jobs, Stages, and Tasks

This is the execution hierarchy — how a single action on a DataFrame becomes thousands of individual units of work:

![Job → Stages → Tasks](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-job-stages-tasks.svg)

| Level | Triggered by | Description |
|---|---|---|
| **Job** | An action (`.count()`, `.write()`, `.show()`) | The top-level unit of work; one action = one job |
| **Stage** | A shuffle boundary | A group of transformations that can run without exchanging data between partitions |
| **Task** | Each partition in a stage | The smallest unit; runs on one executor core, processes one partition |

### Shuffle Boundaries
A **shuffle** is when data has to be redistributed across the network — for example, a `groupBy`, `join`, or `repartition`. Sparks cuts a new stage at every shuffle because all tasks in the upstream stage must complete before the downstream stage can start. Minimising shuffles is one of the highest-leverage Spark optimisations.

## Memory Model

Each executor's heap is divided into regions under the **Unified Memory Manager** (introduced in Spark 1.6):

```
┌────────────────────────────────────────────────┐
│              Executor JVM Heap                 │
├────────────────────────────┬───────────────────┤
│      Unified Memory (60%)  │  Reserved (40%)   │
│  ┌────────────┬──────────┐ │  user code,       │
│  │ Execution  │ Storage  │ │  Spark internals  │
│  │ (shuffles, │ (cache,  │ │                   │
│  │  joins,    │  bcast)  │ │                   │
│  │  sorts)    │          │ │                   │
│  └────────────┴──────────┘ │                   │
│   ↑ can borrow from each   │                   │
└────────────────────────────┴───────────────────┘
```

- **Execution memory** — used during task execution: sort buffers, hash tables for joins, shuffle write buffers. Can spill to disk.
- **Storage memory** — used for cached RDDs/DataFrames and broadcast variables. Evicted using LRU when execution needs space.
- The boundary between them is **soft** — either region can borrow from the other when idle.

Key configs:
| Config | Default | Effect |
|---|---|---|
| `spark.memory.fraction` | `0.6` | Fraction of heap given to unified memory |
| `spark.memory.storageFraction` | `0.5` | Fraction of unified memory reserved for storage (protects cached data from eviction) |

## Local Mode vs. Cluster Mode

| | Local Mode | Cluster Mode |
|---|---|---|
| **What runs where** | Driver and executors in a single JVM on your machine | Driver on one node, executors on worker nodes |
| **Parallelism** | Limited to your laptop's cores (`local[*]` uses all) | Scales to thousands of cores |
| **Use case** | Development, testing, learning | Production workloads |
| **SparkSession** | `.master("local[*]")` | `.master("yarn")` / managed by platform |

Local mode is what most notebooks use — the `local[*]` master string tells Spark to simulate a cluster on the local machine using all available CPU cores.

## Hands-On: Observing the Architecture

Let's start a session and inspect how Spark exposes architecture information at runtime.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkArchitecture") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print(f"App name  : {sc.appName}")
print(f"Master    : {sc.master}")
print(f"Spark UI  : {sc.uiWebUrl}")
print(f"Default parallelism (cores): {sc.defaultParallelism}")

In [ ]:
# Create a DataFrame and force a shuffle (groupBy triggers a stage boundary)
from pyspark.sql.functions import count

df = spark.range(0, 1_000_000)  # 1M rows, partitioned across cores

print(f"Initial partitions: {df.rdd.getNumPartitions()}")

# This triggers 2 stages: map (Stage 1) → shuffle + aggregate (Stage 2)
result = df.groupBy((df.id % 10).alias("bucket")).agg(count("*").alias("cnt"))
result.show()

In [ ]:
# Inspect the execution plan — look for 'Exchange' which marks the shuffle stage boundary
result.explain(mode="formatted")

In [ ]:
# Check how many partitions exist after the shuffle
# spark.sql.shuffle.partitions defaults to 200
print(f"Post-shuffle partitions: {result.rdd.getNumPartitions()}")
print(f"shuffle.partitions config: {spark.conf.get('spark.sql.shuffle.partitions')}")

# For small datasets, 200 shuffle partitions is way too many — tune it down
spark.conf.set("spark.sql.shuffle.partitions", "4")
result2 = df.groupBy((df.id % 10).alias("bucket")).agg(count("*").alias("cnt"))
print(f"After tuning: {result2.rdd.getNumPartitions()} shuffle partitions")

## Summary

- A Spark application has three components: the **Driver**, the **Cluster Manager**, and **Executors** on worker nodes.
- The **Driver** runs your code, builds the DAG, and schedules tasks — it is the single point of coordination.
- The **Cluster Manager** (YARN, Kubernetes, Standalone) allocates resources; Spark is agnostic about which one is used.
- **Executors** are long-lived JVM processes that run tasks and hold cached data.
- An **action** triggers a **Job** → split into **Stages** at shuffle boundaries → each stage runs as parallel **Tasks** (one per partition).
- Executor memory is split into **execution** (joins, sorts, shuffles) and **storage** (cache) regions that can borrow from each other.

Next up: **Setting Up PySpark & Databricks** — getting a local environment running and navigating Databricks.